In [8]:
pip install whisper-at


  Using cached whisper_at-0.5-py3-none-any.whl.metadata (20 kB)
INFO: pip is looking at multiple versions of whisper-at to determine which version is compatible with other requirements. This could take a while.
  Using cached whisper_at-0.4-py3-none-any.whl.metadata (20 kB)
  Using cached whisper_at-0.2-py3-none-any.whl.metadata (18 kB)
  Using cached whisper_at-0.1-py3-none-any.whl.metadata (16 kB)
ERROR: Cannot install whisper-at==0.1, whisper-at==0.2, whisper-at==0.4 and whisper-at==0.5 because these package versions have conflicting dependencies.

The conflict is caused by:
    whisper-at 0.5 depends on triton==2.0.0
    whisper-at 0.4 depends on triton==2.0.0
    whisper-at 0.2 depends on triton==2.0.0
    whisper-at 0.1 depends on triton==2.0.0

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en

In [9]:
pip install --no-deps whisper-at


  Using cached whisper_at-0.5-py3-none-any.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.1 MB/s eta 0:00:00


load model

In [11]:
import whisper_at as whisper
print("✅ Whisper-AT installé avec succès !")

# Test avec le modèle medium
model = whisper.load_model("medium")
print("✅ Modèle chargé - tout fonctionne !")

✅ Whisper-AT installé avec succès !


100%|██████████████████████████████████████| 1.42G/1.42G [00:09<00:00, 161MiB/s]
100%|██████████████████████████████████████| 98.2M/98.2M [00:00<00:00, 103MiB/s]


✅ Modèle chargé - tout fonctionne !


Test avec un audio quelconque

In [15]:
import os
import whisper_at as whisper

# Supprimer l'ancien fichier vide
if os.path.exists("Obama_Yes_we_can.wav"):
    os.remove("Obama_Yes_we_can.wav")
    print("✅ Ancien fichier vide supprimé")

# Option 1 : Télécharger un exemple audio valide (discours JFK)
print("Téléchargement d'un exemple audio...")
!wget -O sample_audio.wav "https://www.signalogic.com/melp/engines/ITU-T_示例_语音/ITU-T_Male_Speech_English_Sample.wav"

# Option 2 : Alternative (fichier de test)
if not os.path.exists("sample_audio.wav") or os.path.getsize("sample_audio.wav") == 0:
    print("Téléchargement avec une autre source...")
    !wget -O sample_audio.wav "https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0010_8k.wav"

# Vérifier que le fichier n'est pas vide
file_size = os.path.getsize("sample_audio.wav")
print(f"✅ Fichier téléchargé : {file_size} bytes")

# Utiliser ce fichier
audio_path = "sample_audio.wav"

# Charger le modèle
print("Chargement du modèle...")
model = whisper.load_model("base")
print("✅ Modèle chargé")

# Traiter l'audio
print("Traitement audio...")
result = model.transcribe(audio_path, at_time_res=2.0)

# Afficher les résultats
audio_embeddings = result['audio_tag']
print(f"\n✅ Nombre de segments de 2s : {audio_embeddings.shape[0]}")
print(f"✅ Dimension des embeddings : {audio_embeddings.shape[1]}")

# Afficher les tags pour chaque segment
from whisper_at import parse_at_label

for i in range(min(5, audio_embeddings.shape[0])):
    start = i * 2
    end = (i + 1) * 2
    tags = parse_at_label(result, top_k=3, p_threshold=0.0)[i]['audio tags']
    print(f"\n📊 Segment [{start}-{end}s]:")
    for tag_name, score in tags:
        print(f"   - {tag_name} (confiance: {score:.3f})")

✅ Ancien fichier vide supprimé
Téléchargement d'un exemple audio...
--2026-04-16 22:08:17--  https://www.signalogic.com/melp/engines/ITU-T_%E7%A4%BA%E4%BE%8B_%E8%AF%AD%E9%9F%B3/ITU-T_Male_Speech_English_Sample.wav
Resolving www.signalogic.com (www.signalogic.com)... 209.150.126.178
Connecting to www.signalogic.com (www.signalogic.com)|209.150.126.178|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-16 22:08:17 ERROR 404: Not Found.

Téléchargement avec une autre source...
--2026-04-16 22:08:17--  https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0010_8k.wav
Resolving www.voiptroubleshooter.com (www.voiptroubleshooter.com)... 162.241.218.124
Connecting to www.voiptroubleshooter.com (www.voiptroubleshooter.com)|162.241.218.124|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 538014 (525K) [audio/x-wav]
Saving to: ‘sample_audio.wav’

sample_audio.wav    100%[===================>] 525.40K  2.97MB/s    in 0.2s    

20

/tmp/ipykernel_20100/692359135.py:32: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result = model.transcribe(audio_path, at_time_res=2.0)



✅ Nombre de segments de 2s : 17
✅ Dimension des embeddings : 527

📊 Segment [0-2s]:
   - Speech (confiance: 1.617)

📊 Segment [2-4s]:
   - Speech (confiance: 2.098)

📊 Segment [4-6s]:
   - Speech (confiance: 1.288)

📊 Segment [6-8s]:
   - Speech (confiance: 2.138)

📊 Segment [8-10s]:
   - Speech (confiance: 0.588)


In [16]:
from whisper_at import parse_at_label
import pandas as pd

# Obtenir tous les tags pour tous les segments
all_tags = parse_at_label(result, top_k=5, p_threshold=0.0)

# Créer un DataFrame pour visualisation
data = []
for i, segment in enumerate(all_tags):
    start = i * 2
    end = (i + 1) * 2
    tags_dict = {tag: score for tag, score in segment['audio tags']}
    tags_dict['segment_id'] = i
    tags_dict['start'] = start
    tags_dict['end'] = end
    data.append(tags_dict)

df = pd.DataFrame(data)
print(df.head(10))

     Speech  segment_id  start  end  Silence
0  1.616673           0      0    2      NaN
1  2.098065           1      2    4      NaN
2  1.287949           2      4    6      NaN
3  2.138211           3      6    8      NaN
4  0.587558           4      8   10      NaN
5  0.825029           5     10   12      NaN
6  1.208684           6     12   14      NaN
7  0.195423           7     14   16      NaN
8  1.164611           8     16   18      NaN
9  1.010980           9     18   20      NaN


In [40]:
# Télécharger un exemple avec de la musique et des sons variés
!wget -O varied_audio.wav "https://samplelib.com/lib/preview/wav/sample-3s.wav"

# Ou mieux : créez un audio avec différents types de sons
# (vous pouvez mixer plusieurs fichiers)

result_varied = model.transcribe("varied_audio.wav", at_time_res=2.0)
embeddings_varied = result_varied['audio_tag']

# Afficher les tags pour chaque segment
for i in range(min(10, embeddings_varied.shape[0])):
    tags = parse_at_label(result_varied, top_k=3, p_threshold=0.0)[i]['audio tags']
    print(f"Segment {i*2}-{(i+1)*2}s: {tags}")

--2026-04-17 00:04:45--  https://samplelib.com/lib/preview/wav/sample-3s.wav
Resolving samplelib.com (samplelib.com)... 188.227.59.182
Connecting to samplelib.com (samplelib.com)|188.227.59.182|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://samplelib.com/wav/sample-3s.wav [following]
--2026-04-17 00:04:45--  https://samplelib.com/wav/sample-3s.wav
Reusing existing connection to samplelib.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 563756 (551K) [audio/x-wav]
Saving to: ‘varied_audio.wav’

varied_audio.wav    100%[===================>] 550.54K  2.03MB/s    in 0.3s    

2026-04-17 00:04:45 (2.03 MB/s) - ‘varied_audio.wav’ saved [563756/563756]



/tmp/ipykernel_20100/587266155.py:7: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result_varied = model.transcribe("varied_audio.wav", at_time_res=2.0)


Segment 0-2s: [('Music', 0.9289692640304565)]
Segment 2-4s: []


In [41]:
tags = parse_at_label(result_varied, top_k=10, p_threshold=-3.0)[i]['audio tags']
print(f"Tous les tags détectés : {tags}")

Tous les tags détectés : [('Music', -0.045404572039842606)]


*******************************************************************************************


Tests avec des audio piano, parole, musique+parole

In [38]:
print("📥 Téléchargement d'exemples audio clairs...\n")

# 1. Musique pure (piano)
!wget -q -O music_piano.wav "https://www.kozco.com/tech/piano2.wav"
print("✅ 1. Musique piano téléchargée")

# 3. Parole seule (discours)
!wget -q -O speech_only.wav "https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0010_8k.wav"
print("✅ 2. Parole seule téléchargée")

# 4. Créer un mix (musique + parole) - Optionnel
print("\n🎵 Création d'un mix musique+parole...")
!ffmpeg -i music_piano.wav -i speech_only.wav -filter_complex amix=inputs=2:duration=longest mixed_audio.wav -y 2>/dev/null
print("✅ 3. Mix musique+parole créé")

print("\n" + "="*60)
print("🔬 ANALYSE DES RÉSULTATS")
print("="*60 + "\n")


📥 Téléchargement d'exemples audio clairs...

✅ 1. Musique piano téléchargée
✅ 2. Parole seule téléchargée

🎵 Création d'un mix musique+parole...
✅ 3. Mix musique+parole créé

🔬 ANALYSE DES RÉSULTATS



In [21]:
def analyze_audio(file_path, name, time_res=2.0):
    """Analyse un fichier audio et affiche les tags détectés"""
    print(f"\n{'─'*50}")
    print(f"📁 TEST : {name}")
    print(f"📄 Fichier : {file_path}")
    print(f"{'─'*50}")

    # Transcrire
    result = model.transcribe(file_path, at_time_res=time_res)
    embeddings = result['audio_tag']

    print(f"📊 Durée totale : {embeddings.shape[0] * time_res:.1f} secondes")
    print(f"📊 Nombre de segments de {time_res}s : {embeddings.shape[0]}")

    # Analyser chaque segment
    for i in range(embeddings.shape[0]):
        start = i * time_res
        end = (i + 1) * time_res

        # Obtenir les tags avec seuil bas
        tags = parse_at_label(result, top_k=5, p_threshold=-3.0)[i]['audio tags']

        print(f"\n  🎬 Segment [{start:.1f}-{end:.1f}s]:")
        if tags:
            for tag_name, score in tags:
                # Marquer les scores positifs avec 🟢
                marker = "🟢" if score > 0 else "🟡"
                print(f"     {marker} {tag_name}: {score:.3f}")
        else:
            print(f"     ⚪ Aucun tag détecté (fichier trop court ou silencieux)")

    return result

In [23]:
# Test 1 : Musique piano
result_music = analyze_audio("music_piano.wav", "MUSIQUE PIANO", time_res=2.0)


# Test 3 : Parole seule
result_speech = analyze_audio("speech_only.wav", "PAROLE SEULE", time_res=2.0)

# Test 4 : Mix musique + parole
result_mixed = analyze_audio("mixed_audio.wav", "MIX (MUSIQUE + PAROLE)", time_res=2.0)


──────────────────────────────────────────────────
📁 TEST : MUSIQUE PIANO
📄 Fichier : music_piano.wav
──────────────────────────────────────────────────


/tmp/ipykernel_20100/3666776601.py:9: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result = model.transcribe(file_path, at_time_res=time_res)


📊 Durée totale : 8.0 secondes
📊 Nombre de segments de 2.0s : 4

  🎬 Segment [0.0-2.0s]:
     🟢 Music: 0.223
     🟡 Electric piano: -0.377
     🟡 Keyboard (musical): -0.695
     🟡 Piano: -1.324
     🟡 Electronic tuner: -1.835

  🎬 Segment [2.0-4.0s]:
     🟢 Music: 0.784
     🟢 Electric piano: 0.025
     🟡 Piano: -0.240
     🟡 Keyboard (musical): -0.534
     🟡 Musical instrument: -1.770

  🎬 Segment [4.0-6.0s]:
     🟡 Music: -0.143
     🟡 Keyboard (musical): -1.295
     🟡 Piano: -1.392
     🟡 Electric piano: -1.412
     🟡 Musical instrument: -1.801

  🎬 Segment [6.0-8.0s]:
     🟢 Silence: 0.239
     🟢 Music: 0.191

──────────────────────────────────────────────────
📁 TEST : PAROLE SEULE
📄 Fichier : speech_only.wav
──────────────────────────────────────────────────


/tmp/ipykernel_20100/3666776601.py:9: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result = model.transcribe(file_path, at_time_res=time_res)


📊 Durée totale : 34.0 secondes
📊 Nombre de segments de 2.0s : 17

  🎬 Segment [0.0-2.0s]:
     🟢 Speech: 1.617
     🟡 Narration, monologue: -1.234
     🟡 Female speech, woman speaking: -1.619

  🎬 Segment [2.0-4.0s]:
     🟢 Speech: 2.098
     🟡 Female speech, woman speaking: -1.885
     🟡 Knock: -2.119
     🟡 Stomach rumble: -2.347
     🟡 Animal: -2.702

  🎬 Segment [4.0-6.0s]:
     🟢 Speech: 1.288
     🟡 Inside, small room: -2.588
     🟡 Narration, monologue: -2.929

  🎬 Segment [6.0-8.0s]:
     🟢 Speech: 2.138
     🟡 Narration, monologue: -1.903
     🟡 Female speech, woman speaking: -1.974
     🟡 Inside, small room: -2.915

  🎬 Segment [8.0-10.0s]:
     🟢 Speech: 0.588
     🟡 Narration, monologue: -1.403
     🟡 Female speech, woman speaking: -1.827

  🎬 Segment [10.0-12.0s]:
     🟢 Speech: 0.825
     🟡 Inside, small room: -2.576
     🟡 Narration, monologue: -2.704
     🟡 Zipper (clothing): -2.799
     🟡 Female speech, woman speaking: -2.942

  🎬 Segment [12.0-14.0s]:
     🟢 Speech: 1

/tmp/ipykernel_20100/3666776601.py:9: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result = model.transcribe(file_path, at_time_res=time_res)


📊 Durée totale : 34.0 secondes
📊 Nombre de segments de 2.0s : 17

  🎬 Segment [0.0-2.0s]:
     🟢 Music: 1.013
     🟢 Speech: 0.486
     🟡 Piano: -1.861
     🟡 Keyboard (musical): -2.171
     🟡 Musical instrument: -2.389

  🎬 Segment [2.0-4.0s]:
     🟢 Music: 1.448
     🟢 Speech: 0.796
     🟡 Musical instrument: -1.072
     🟡 Piano: -1.368
     🟡 Keyboard (musical): -1.683

  🎬 Segment [4.0-6.0s]:
     🟢 Music: 0.898
     🟢 Speech: 0.119

  🎬 Segment [6.0-8.0s]:
     🟢 Speech: 1.598
     🟡 Female speech, woman speaking: -1.896
     🟡 Narration, monologue: -2.096
     🟡 Inside, small room: -2.925

  🎬 Segment [8.0-10.0s]:
     🟢 Speech: 0.702
     🟡 Narration, monologue: -1.317
     🟡 Female speech, woman speaking: -1.612

  🎬 Segment [10.0-12.0s]:
     🟢 Speech: 0.747
     🟡 Inside, small room: -2.865
     🟡 Narration, monologue: -2.913

  🎬 Segment [12.0-14.0s]:
     🟢 Speech: 1.073
     🟡 Narration, monologue: -1.680
     🟡 Inside, small room: -2.574
     🟡 Female speech, woman speaki

In [36]:
print("\n" + "="*60)
print("📋 RÉSUMÉ DES DÉTECTIONS")
print("="*60)

def get_top_tags(result, top_n=3):
    """Extrait les tags les plus fréquents"""
    all_tags = []
    for i in range(result['audio_tag'].shape[0]):
        tags = parse_at_label(result, top_k=5, p_threshold=-3.0)[i]['audio tags']
        all_tags.extend([t[0] for t in tags if t[1] > -1.0])
    return list(dict.fromkeys(all_tags))[:top_n]

print(f"\n🎵 Musique piano : {get_top_tags(result_music)}")
print(f"🗣️ Parole seule : {get_top_tags(result_speech)}")
print(f"🎹+🗣️ Mix : {get_top_tags(result_mixed)}")


📋 RÉSUMÉ DES DÉTECTIONS

🎵 Musique piano : ['Music', 'Electric piano', 'Keyboard (musical)']
🗣️ Parole seule : ['Speech', 'Silence', 'Music']
🎹+🗣️ Mix : ['Pink noise', 'White noise']


/usr/local/lib/python3.12/dist-packages/whisper_at/at_post_processing.py:34: UserWarning: nn language not supported. Use English label names instead. If you wish to use label names of a specific language, please specify the language argument
  warnings.warn("{:s} language not supported. Use English label names instead. If you wish to use label names of a specific language, please specify the language argument".format(language))


Test avec son applaudissement

In [35]:
from google.colab import files
import whisper_at as whisper
from whisper_at import parse_at_label
import os

print("📤 Upload de votre fichier MP3 d'applaudissements")
print("="*50)

# Uploader le fichier
uploaded = files.upload()

# Récupérer le nom du fichier uploadé
for filename in uploaded.keys():
    print(f"\n✅ Fichier uploadé : {filename}")
    fichier_mp3 = filename
    break

# Convertir et analyser
fichier_wav = "uploaded_applause.wav"
print(f"\n🔄 Conversion en WAV...")
!ffmpeg -i {fichier_mp3} -ar 16000 -ac 1 {fichier_wav} -y 2>/dev/null

if os.path.exists(fichier_wav) and os.path.getsize(fichier_wav) > 10000:
    print("✅ Conversion réussie\n")

    # Analyser
    result = model.transcribe(fichier_wav, at_time_res=2.0)
    embeddings = result['audio_tag']

    print(f"📊 {embeddings.shape[0]} segments de 2 secondes\n")

    # Afficher les tags
    for i in range(embeddings.shape[0]):
        start = i * 2
        end = (i + 1) * 2
        tags = parse_at_label(result, top_k=5, p_threshold=-2.0)[i]['audio tags']

        print(f"Segment [{start}-{end}s]:")
        for tag_name, score in tags:
            if 'applause' in tag_name.lower():
                print(f"  🎯 {tag_name}: {score:.3f} ← APPLAUDISSEMENT !")
            else:
                print(f"     {tag_name}: {score:.3f}")
        print()
else:
    print("❌ Erreur lors de la conversion")

📤 Upload de votre fichier MP3 d'applaudissements


Saving vvqne-applause-383901.mp3 to vvqne-applause-383901.mp3

✅ Fichier uploadé : vvqne-applause-383901.mp3

🔄 Conversion en WAV...
✅ Conversion réussie



/tmp/ipykernel_20100/2755461124.py:27: UserWarning: Current at_time_res is 2.00 second, the audio tagging model is trained with time resolution of 10 seconds. Mismatch time resolution may cause an audio tagging performance drop, but won't impact ASR performance.
  result = model.transcribe(fichier_wav, at_time_res=2.0)


📊 14 segments de 2 secondes

Segment [0-2s]:
  🎯 Applause: 1.629 ← APPLAUDISSEMENT !

Segment [2-4s]:
  🎯 Applause: 2.097 ← APPLAUDISSEMENT !

Segment [4-6s]:
  🎯 Applause: 1.767 ← APPLAUDISSEMENT !

Segment [6-8s]:
  🎯 Applause: 1.733 ← APPLAUDISSEMENT !

Segment [8-10s]:
  🎯 Applause: 1.241 ← APPLAUDISSEMENT !

Segment [10-12s]:
  🎯 Applause: 0.652 ← APPLAUDISSEMENT !

Segment [12-14s]:
  🎯 Applause: 0.750 ← APPLAUDISSEMENT !

Segment [14-16s]:
  🎯 Applause: -0.201 ← APPLAUDISSEMENT !

Segment [16-18s]:
  🎯 Applause: 0.035 ← APPLAUDISSEMENT !

Segment [18-20s]:
  🎯 Applause: -1.060 ← APPLAUDISSEMENT !

Segment [20-22s]:
     Silence: -0.717

Segment [22-24s]:
     Silence: 0.750
     Speech: -1.122

Segment [24-26s]:
     Silence: 0.042
     Speech: -1.786

Segment [26-28s]:
     Silence: -1.647

